# Validate Modeling Table

This notebook validates that the modeling mart produced by dbt (main.mart_grad_rate_risk) is ready for supervising

Validation focuses on:
 - Table availability & schema sanity
 - Target distribution (class balance/imbalance)
 - Feature coverage (null rates, range checks, categorical levels)
 - Baseline expectation (majority-class accruacy)

 The goal is to establish confidence that the dataset is usable to document modeling constraints before training

In [2]:
import duckdb
import pandas as pd

DB_PATH = "../accountability_project/dev.duckdb"

con = duckdb.connect(DB_PATH)

## 1. Confirm mart exists and inspect schema

In [3]:
con.execute("describe main.mart_grad_rate_risk").fetch_df()

,column_name,column_type,null,key,default,extra
0,district_number,VARCHAR,YES,None,None,None
1,district_name,VARCHAR,YES,None,None,None
2,school_number,VARCHAR,YES,None,None,None
3,school_name,VARCHAR,YES,None,None,None
4,school_type,INTEGER,YES,None,None,None
5,title_i,VARCHAR,YES,None,None,None
6,pct_econ_disadvantaged,DOUBLE,YES,None,None,None
7,grade_2025,VARCHAR,YES,None,None,None
8,grade_2024,VARCHAR,YES,None,None,None
9,grade_2025_score,INTEGER,YES,None,None,None


## 2. Row count and uniqueness checks

Primary Key expectation (for this project scope): **one row per school**.

Checks:
- total rows
- distinct school count
- duplicates (if any)

In [5]:
row_counts = con.execute("""
select
    count(*) as n_rows,
    count(distinct school_number) as n_distinct_schools,
    (count (*) - count(distinct school_number)) as n_duplicate_school_rows
from main.mart_grad_rate_risk
""").fetch_df()

row_counts

,n_rows,n_distinct_schools,n_duplicate_school_rows
0,3465,1225,2240


## 3. Target distribution (is_grad_rate_risk)

Risk definition used so far:
- is_grad_rate_risk = 1 when graduation_rate_2023_24 < 80
- is_grad_rate_risk = 0 otherwise

This section quantifies:
- class counts
- class percentages
- implied imbalance

In [6]:
target_dist = con.execute("""
select
    is_grad_rate_risk,
    count(*) as n
from main.mart_grad_rate_risk
group by 1
order by 1
""").fetch_df()

target_dist

,is_grad_rate_risk,n
0,0,3445
1,1,20


In [7]:
target_pct = con.execute("""
with base as (
    select is_grad_rate_risk, count(*) as n
    from main.mart_grad_rate_risk
    group by 1                         
),
tot as (
    select sum(n) as total from base                          
)
select
    b.is_grad_rate_risk,
    b.n,
    round(100.0 * b.n / t.total, 2) as pct
from base b
cross join tot t
order by b.is_grad_rate_risk
""").fetch_df()

target_pct

,is_grad_rate_risk,n,pct
0,0,3445,99.42
1,1,20,0.58


## 4. Baseline expectation (majority-class accuracy)

Before training any model, it helps to compute the accruracy of a trivial baseline:
- Predict the most frequent classs for every row.

If the baseline is already extremely high, accuracy alone will not be a meaninful metric (precision/recall, F1, PR-AUC become more important).

In [9]:
baseline = con.execute("""
with base as (
    select is_grad_rate_risk, count(*) as n                       
    from main.mart_grad_rate_risk
    group by 1
),
tot as (
    select sum(n) as total from base                       
)
select
    max(n) as majority_class_n,
    (select total from tot) as total_n,
    round(100.0 * max(n) / (select total from tot), 2) as majority_class_accuracy_pct
from base                       
""").fetch_df()

baseline

,majority_class_n,total_n,majority_class_accuracy_pct
0,3445,3465.0,99.42


In [10]:
con.close()

## Validation summary and modeling implications

The `mart_grad_rate_risk` table is structurally sound and contains the required features and target label for risk modeling.

However:
- The dataset contains multiple rows per school, which may introduce bias or leakage if modeled directly.
- The target is highly imbalanced (≈0.6%), making accuracy an unsuitable evaluation metric.

Before training a classifier, the data should be reshaped to a single row per school (or the modeling objective should explicitly justify repeated rows).
Subsequent modeling should prioritize recall, precision, and PR-AUC rather than accuracy. 